# Taxonomy Construction

In [ ]:
import json
import os
from collections import defaultdict
from typing import Any, Dict, Iterable, List, Set, Tuple

import numpy as np
import pandas as pd

from taxonomy import (
    PathConfig,
    LLMConfig,
    EmbeddingConfig,
    NMFConfig,
    TaxonomyConfig,
    build_chat_model,
    load_from_pickle,
    save_to_pickle,
    generate_embeddings_in_batches,
    build_k_values_by_size,
    recursive_nmf_split_until_threshold,
    flatten_tree_leaves_only,
    run_fine_grained_local_construction,
    run_fine_grained_taxonomy_merge,
    run_broad_local_construction,
    run_broad_taxonomy_merge,
    clean_taxonomy_edges,
    normalize_edges,
    print_hierarchy,
    resolve_transitive_mapping,
)


## Configuration


In [ ]:
# openai_api_key = os.environ.get("OPENAI_API_KEY")

cfg = TaxonomyConfig(
    paths=PathConfig(
        domain="nephrology", 
        folder_name="nephrology",
        data_root="data/processed",
        log_subdir="logs",
        experiment_name="experiment1",
    ),
    llm=LLMConfig(
        provider="vllm",
        base_url="http://localhost:8898/v1",
        model="openai/gpt-oss-20b",
        api_key="EMPTY",
        temperature=0,
        retry_temperature=1,
        top_p=1,
        seed=0,
        timeout=120,
        max_output_tokens=8192,
        max_retries=5,
        retry_delay=1,
        verbose=False,
    ),
    # llm=LLMConfig(
    #     provider="openai",
    #     model="gpt-5.4",
    #     api_key=openai_api_key or None,
    #     temperature=0,
    #     timeout=60,
    #     max_output_tokens=4096,
    #     max_retries=5,
    #     retry_delay=1,
    #     verbose=False,
    # ),
    embedding=EmbeddingConfig(
        model_name="all-mpnet-base-v2",
        batch_size=1024,
        similarity_threshold=0.75,
    ),
    nmf=NMFConfig(
        w_recon=0.40,
        w_sparsity=0.25,
        w_coherence=0.35,
        coherence_floor_ratio=None,
        rel_cutoff=0.2,
        min_terms=1,
        add_singletons=True,
        split_threshold=20,
        k_nn_cap=50,
    ),
)

domain = cfg.paths.domain
file_path = cfg.paths.file_path
experiment_name = cfg.paths.experiment_name
log_dir = os.path.join(cfg.paths.log_subdir, domain, experiment_name)
os.makedirs(log_dir, exist_ok=True)


generation_cfg = cfg
integration_cfg = cfg
generation_chat_model = build_chat_model(generation_cfg.llm)
integration_chat_model = build_chat_model(integration_cfg.llm)


## Load or create inputs


In [ ]:
experiment_dir = os.path.join(file_path, experiment_name)


In [ ]:
keyword_mapping = load_from_pickle(os.path.join(file_path, 'keyword_mapping.pkl'))

keywords = sorted({
    entry["final"]
    for entry in keyword_mapping.values()
    if entry.get("final") is not None
})
keywords = [keyword for keyword in keywords if keyword != ""]

print(len(keywords))

keyword_embeddings = generate_embeddings_in_batches(
    keywords,
    batch_size=cfg.embedding.batch_size,
    embedding_model=cfg.embedding.model_name,
)

## Generate and integrate local hierarchies



In [ ]:
import time

start_time = time.perf_counter()

selected_keywords = keywords
selected_keyword_embeddings = generate_embeddings_in_batches(
    selected_keywords,
    batch_size=cfg.embedding.batch_size,
    embedding_model=cfg.embedding.model_name,
)

base_params = dict(
    k_nn=min(max(5, int(np.sqrt(len(selected_keywords)))), cfg.nmf.k_nn_cap, len(selected_keywords) - 1),
    k_values=build_k_values_by_size(len(selected_keywords)),
    w_recon=cfg.nmf.w_recon,
    w_sparsity=cfg.nmf.w_sparsity,
    w_coherence=cfg.nmf.w_coherence,
    coherence_floor_ratio=cfg.nmf.coherence_floor_ratio,
    min_terms=cfg.nmf.min_terms,
    max_terms=len(selected_keywords),
    rel_cutoff=cfg.nmf.rel_cutoff,
    add_singletons=cfg.nmf.add_singletons,
    show_plots=False,
)

fine_grained_tree = recursive_nmf_split_until_threshold(
    terms=selected_keywords,
    embeddings=selected_keyword_embeddings,
    base_params=base_params,
    split_threshold=cfg.nmf.split_threshold,
)

fine_grained_topics = flatten_tree_leaves_only(fine_grained_tree)
print(len(fine_grained_topics))
for topic in fine_grained_topics:
    print(topic)

os.makedirs(experiment_dir, exist_ok=True)
save_to_pickle(fine_grained_topics, os.path.join(experiment_dir, "fine_grained_topics.pkl"))

elapsed_time = time.perf_counter() - start_time
print(f"Total running time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)")


In [ ]:
local_taxonomies = run_fine_grained_local_construction(
    fine_grained_topics,
    generation_chat_model,
    generation_cfg,
    resume=True,
    log_to_file=True,
    verbose=True,
    enable_telemetry=True,
    llm_concurrency=1, 
)

print(f"Saved {len(local_taxonomies)} local taxonomies")


In [ ]:
# If ManualCycleDecisionRequired is raised, edit the reported JSON file and
# rerun this cell with resume=True.
fine_state = run_fine_grained_taxonomy_merge(
    integration_chat_model,
    integration_cfg,
    resume=True,
    log_to_file=True,
    verbose=True,
    enable_telemetry=True,
)

fine_grained_edges = fine_state.edges
fine_grained_term_mapping = fine_state.term_mapping
evaluated_pairs = fine_state.evaluated_pairs
subsumption_evaluated_pairs = fine_state.subsumption_evaluated_pairs
topic_ids_completed = fine_state.topic_ids_completed

In [ ]:
fine_grained_edges_merged = normalize_edges(
    fine_grained_edges,
    fine_grained_term_mapping,
)
fine_grained_cleaned_edges = clean_taxonomy_edges(
    fine_grained_edges_merged,
    integration_chat_model,
    domain,
    verbose=True,
    connect_to_domain=True,
)
save_to_pickle(
    fine_grained_cleaned_edges,
    os.path.join(experiment_dir, "fine_grained_cleaned_edges.pkl"),
)

print(f"Number of fine-grained cleaned edges: {len(fine_grained_cleaned_edges)}")
print_hierarchy(fine_grained_cleaned_edges)


## Refine the global hierarchy (round 1)

In [ ]:

start_time = time.perf_counter()


sub_roots = sorted({
    child
    for child, parent in fine_grained_cleaned_edges
    if parent == domain.lower()
})
for child in sub_roots:
    print(child)

print(len(sub_roots))
save_to_pickle(sub_roots, os.path.join(experiment_dir, "sub_roots.pkl"))

selected_keywords = sub_roots
selected_keyword_embeddings = generate_embeddings_in_batches(
    selected_keywords,
    batch_size=cfg.embedding.batch_size,
    embedding_model=cfg.embedding.model_name,
)

base_params = dict(
    k_nn=min(max(5, int(np.sqrt(len(selected_keywords)))), cfg.nmf.k_nn_cap, len(selected_keywords) - 1),
    k_values=build_k_values_by_size(len(selected_keywords)),
    w_recon=cfg.nmf.w_recon,
    w_sparsity=cfg.nmf.w_sparsity,
    w_coherence=cfg.nmf.w_coherence,
    coherence_floor_ratio=cfg.nmf.coherence_floor_ratio,
    min_terms=cfg.nmf.min_terms,
    max_terms=len(selected_keywords),
    rel_cutoff=cfg.nmf.rel_cutoff,
    add_singletons=cfg.nmf.add_singletons,
    show_plots=False,
)

coarse_tree = recursive_nmf_split_until_threshold(
    terms=selected_keywords,
    embeddings=selected_keyword_embeddings,
    base_params=base_params,
    split_threshold=10,
)

coarse_topics = flatten_tree_leaves_only(coarse_tree)
print(len(coarse_topics))
for topic in coarse_topics:
    print(topic)

save_to_pickle(coarse_topics, os.path.join(experiment_dir, "coarse_topics.pkl"))

elapsed_time = time.perf_counter() - start_time
print(f"Total running time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)")


In [ ]:
broad_local_taxonomies_1 = run_broad_local_construction(
    coarse_topics,
    generation_chat_model,
    generation_cfg,
    output_suffix="",
    resume=True,
    log_to_file=True,
    verbose=True,
    enable_telemetry=True,
    llm_concurrency=1,
)

print(f"Saved {len(broad_local_taxonomies_1)} broad pass 1 local taxonomies")


In [ ]:
broad_state_1 = run_broad_taxonomy_merge(
    integration_chat_model,
    integration_cfg,
    initial_edges=fine_grained_cleaned_edges,
    initial_mapping=fine_grained_term_mapping,
    linkage_roots=sub_roots,
    output_suffix="",
    initial_evaluated_pairs=fine_state.evaluated_pairs,
    initial_subsumption_evaluated_pairs=fine_state.subsumption_evaluated_pairs,
    resume=True,
    log_to_file=True,
    verbose=True,
    enable_telemetry=True,
)

term_mapping1 = broad_state_1.term_mapping
existing_edges1 = broad_state_1.edges
s_f_roots = broad_state_1.linkage_roots


In [ ]:
existing_cleaned_edges = clean_taxonomy_edges(
    existing_edges1,
    integration_chat_model,
    domain,
    term_mapping=term_mapping1,
    verbose=True,
    connect_to_domain=True,
)
save_to_pickle(
    existing_cleaned_edges,
    os.path.join(experiment_dir, "existing_cleaned_edges.pkl"),
)

print(f"Number of cleaned edges after broad pass 1: {len(existing_cleaned_edges)}")
print_hierarchy(existing_cleaned_edges)


## Refine the global hierarchy (round 2)

In [ ]:

start_time = time.perf_counter()

sub_roots2 = sorted({
    child
    for child, parent in existing_cleaned_edges
    if parent == domain.lower()
})
for child in sub_roots2:
    print(child)

print(len(sub_roots2))
save_to_pickle(sub_roots2, os.path.join(experiment_dir, "sub_roots2.pkl"))

selected_keywords = sub_roots2
selected_keyword_embeddings = generate_embeddings_in_batches(
    selected_keywords,
    batch_size=cfg.embedding.batch_size,
    embedding_model=cfg.embedding.model_name,
)

base_params = dict(
    k_nn=min(max(5, int(np.sqrt(len(selected_keywords)))), cfg.nmf.k_nn_cap, len(selected_keywords) - 1),
    k_values=build_k_values_by_size(len(selected_keywords)),
    w_recon=cfg.nmf.w_recon,
    w_sparsity=cfg.nmf.w_sparsity,
    w_coherence=cfg.nmf.w_coherence,
    coherence_floor_ratio=cfg.nmf.coherence_floor_ratio,
    min_terms=cfg.nmf.min_terms,
    max_terms=len(selected_keywords),
    rel_cutoff=cfg.nmf.rel_cutoff,
    add_singletons=cfg.nmf.add_singletons,
    show_plots=False,
)

coarse_tree2 = recursive_nmf_split_until_threshold(
    terms=selected_keywords,
    embeddings=selected_keyword_embeddings,
    base_params=base_params,
    split_threshold=5,
)

coarse_topics2 = flatten_tree_leaves_only(coarse_tree2)
print(len(coarse_topics2))
for topic in coarse_topics2:
    print(topic)

save_to_pickle(coarse_topics2, os.path.join(experiment_dir, "coarse_topics2.pkl"))

elapsed_time = time.perf_counter() - start_time
print(f"Total running time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)")

In [ ]:
broad_local_taxonomies_2 = run_broad_local_construction(
    coarse_topics2,
    generation_chat_model,
    generation_cfg,
    output_suffix="2",
    resume=True,
    log_to_file=True,
    verbose=True,
    enable_telemetry=True,
    llm_concurrency=1,
)

print(f"Saved {len(broad_local_taxonomies_2)} broad pass 2 local taxonomies")


In [ ]:
broad_state_2 = run_broad_taxonomy_merge(
    integration_chat_model,
    integration_cfg,
    initial_edges=existing_cleaned_edges,
    initial_mapping=term_mapping1,
    linkage_roots=sub_roots2,
    output_suffix="2",
    initial_evaluated_pairs=broad_state_1.evaluated_pairs,
    initial_subsumption_evaluated_pairs=broad_state_1.subsumption_evaluated_pairs,
    resume=True,
    log_to_file=True,
    verbose=True,
    enable_telemetry=True,
)


term_mapping2 = broad_state_2.term_mapping
existing_edges2 = broad_state_2.edges
s_f_roots2 = broad_state_2.linkage_roots


In [ ]:
existing_cleaned_edges2 = clean_taxonomy_edges(
    existing_edges2,
    integration_chat_model,
    domain,
    term_mapping=term_mapping2,
    verbose=True,
    connect_to_domain=True,
)
save_to_pickle(
    existing_cleaned_edges2,
    os.path.join(experiment_dir, "existing_cleaned_edges2.pkl"),
)

print(f"Number of cleaned edges after broad pass 2: {len(existing_cleaned_edges2)}")
print_hierarchy(existing_cleaned_edges2)


## Final taxonomy export

In [ ]:
def augment_keywords_with_descendants(
    all_edges: Iterable[Tuple[str, str]],
    dict_topic_keywords: Dict[str, List[Any]],
) -> tuple[dict[str, list[Any]], dict[Any, set[str]]]:
    children_of: Dict[str, Set[str]] = defaultdict(set)
    nodes: Set[str] = set(dict_topic_keywords.keys())

    for child, parent in all_edges:
        children_of[parent].add(child)
        nodes.add(child)
        nodes.add(parent)

    memo: Dict[str, Set[Any]] = {}

    def gather(topic: str) -> Set[Any]:
        if topic in memo:
            return memo[topic]

        acc: Set[Any] = set(dict_topic_keywords.get(topic, []))
        for child in children_of.get(topic, ()):
            acc.update(gather(child))

        memo[topic] = acc
        return acc

    dict_t_ks_aggr = {topic: list(gather(topic)) for topic in nodes}
    dict_k_ts_aggr = defaultdict(set)
    for topic, keywords_for_topic in dict_t_ks_aggr.items():
        for keyword in keywords_for_topic:
            dict_k_ts_aggr[keyword].add(topic)

    return dict_t_ks_aggr, dict_k_ts_aggr




In [ ]:
dict_keyword_to_topic2 = defaultdict()
dict_topic_to_keywords2 = defaultdict(list)
for raw_keyword in keywords:
    topic = resolve_transitive_mapping(raw_keyword.lower(), term_mapping2)
    dict_keyword_to_topic2[raw_keyword] = topic
    dict_topic_to_keywords2[topic].append(raw_keyword)

existing_cleaned_edges_selected = existing_cleaned_edges2.copy()
dict_topic_to_keywords_selected = dict(dict_topic_to_keywords2)

dict_topic_keywords_aggr, dict_keyword_topics_aggr = augment_keywords_with_descendants(
    existing_cleaned_edges_selected,
    dict_topic_to_keywords_selected,
)

topics_with_no_keywords = [
    topic for topic, topic_keywords in dict_topic_keywords_aggr.items()
    if len(topic_keywords) == 0
]

existing_cleaned_edges_selected = [
    edge for edge in existing_cleaned_edges_selected
    if edge[0] not in topics_with_no_keywords and edge[1] not in topics_with_no_keywords
]

existing_cleaned_nodes_selected = sorted(
    {node for edge in existing_cleaned_edges_selected for node in edge}
)
dict_topic_to_keywords_selected = {
    topic: topic_keywords
    for topic, topic_keywords in dict_topic_to_keywords_selected.items()
    if topic not in topics_with_no_keywords
}
dict_topic_to_keywords_selected = {
    topic: topic_keywords
    for topic, topic_keywords in dict_topic_to_keywords_selected.items()
    if topic in existing_cleaned_nodes_selected
}

dict_keyword_to_topic_selected = {
    keyword: topic
    for topic, topic_keywords in dict_topic_to_keywords_selected.items()
    for keyword in topic_keywords
}
print(f"Final term-to-node mappings: {len(dict_keyword_to_topic_selected)}")

final_taxonomy_dir = os.path.join(
    "results",
    cfg.paths.folder_name,
    experiment_name,
)
os.makedirs(final_taxonomy_dir, exist_ok=True)

json_outputs = {
    "edges.json": existing_cleaned_edges_selected,
    "nodes.json": existing_cleaned_nodes_selected,
    "term_to_node_mappings.json": dict_keyword_to_topic_selected,
}
for filename, data in json_outputs.items():
    output_path = os.path.join(final_taxonomy_dir, filename)
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
        file.write("\n")
    print(f"Saved {output_path}")

final_taxonomy_path = os.path.join(final_taxonomy_dir, "final_taxonomy.txt")
print_hierarchy(existing_cleaned_edges_selected, final_taxonomy_path)
print(
    f"Final taxonomy summary: {len(existing_cleaned_edges_selected)} edges, "
    f"{len(existing_cleaned_nodes_selected)} nodes"
)
